In [15]:
import pandas as pd
import os
import glob

# Path to the master parquet file
master_df_path = '/home/chb3333/yulab/chb3333/gem-patho/data_extraction/patient_mutation_polyphen_vector_OStime/master_df.parquet'

In [16]:
# Load the DataFrame
df = pd.read_parquet(master_df_path)

In [17]:
def check_file_availability(row):
    # Adjust these column names if needed
    project_id = row['Project ID']  # e.g., 'TCGA-ESCA'
    case_id = row['Case ID']        # e.g., 'TCGA-V5-A7RB'
    
    # Construct both directory paths:
    pm_folder = "/n/data2/hms/dbmi/kyu/lab/NCKU/foundation_model_features/WSI_features/{0}-PM/GIGAPATH/20X/pt_files(stain_norm)".format(project_id)
    fs_folder = "/n/data2/hms/dbmi/kyu/lab/NCKU/foundation_model_features/WSI_features/{0}-FS/GIGAPATH/20X/pt_files(stain_norm)".format(project_id)
    
    # Create glob patterns: the file starts with the Case ID and ends with .pt
    pattern_pm = os.path.join(pm_folder, "{}*.pt".format(case_id))
    pattern_fs = os.path.join(fs_folder, "{}*.pt".format(case_id))
    
    # Check existence in both folders
    pm_exists = len(glob.glob(pattern_pm)) > 0
    fs_exists = len(glob.glob(pattern_fs)) > 0
    
    return pd.Series({'PM': pm_exists, 'FS': fs_exists})

In [18]:
# Apply the function to each row to determine file availability in each folder
availability = df.apply(check_file_availability, axis=1)
df = pd.concat([df, availability], axis=1)

# Calculate percentages
total_cases = len(df)

# Cases with no FS available (i.e., FS is False)
no_fs_cases = df[~df['FS']]
perc_no_fs = (len(no_fs_cases) / total_cases) * 100

# Cases with no PM available (i.e., PM is False)
no_pm_cases = df[~df['PM']]
perc_no_pm = (len(no_pm_cases) / total_cases) * 100

# Cases with nothing available (neither PM nor FS)
no_files_cases = df[(df['PM'] == False) & (df['FS'] == False)]
perc_no_files = (len(no_files_cases) / total_cases) * 100

print("Total cases: {}".format(total_cases))
print("Percentage of cases with no FS file available: {:.2f}%".format(perc_no_fs))
print("Percentage of cases with no PM file available: {:.2f}%".format(perc_no_pm))
print("Percentage of cases with no file available in either folder: {:.2f}%".format(perc_no_files))


Total cases: 10190
Percentage of cases with no FS file available: 3.21%
Percentage of cases with no PM file available: 15.11%
Percentage of cases with no file available in either folder: 2.49%


In [ ]:
if not no_files_cases.empty:
    print("\nCases with no available file in either folder:")
    print(no_files_cases[['Project ID', 'Case ID']])

In [23]:
# Optionally, save the updated DataFrame with the new columns
output_path = '/home/chb3333/yulab/chb3333/gem-patho/data_extraction/image_extraction/master_df_with_file_check.csv'
df.to_csv(output_path, index=False)